# 02. Отбор признаков для регрессии

**Цель**: выбрать подмножество из 23 признаков, минимизирующее мультиколлинеарность, сохраняя сигнал для `ln(price)`.

**Алгоритм**:
1. Иерархическая кластеризация признаков по `1 − |corr|`. Признаки в одном кластере → попарная `|corr| ≥ 0.80` (двойники).
2. В каждом кластере **champion** = признак с максимальной `|corr|` с `ln(price)`.
3. Двойники → DROP. Одиночка-чемпион → DROP только если `corr_y < 0.15` И communality_10PC < 0.90 (бесполезен и для Y, и для X).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))

from helpers import (set_plot_style, load_data, impute_median, NUM_COLS_FULL)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

set_plot_style()
df, df_train, df_test = load_data()
df_train_i = impute_median(df_train, NUM_COLS_FULL)


## 1. Кластеризация признаков по `1 − |corr|`

Дендрограмма показывает, какие признаки объединяются в пары/группы по корреляции. Горизонтальная линия на уровне 0.15 = порог `|corr| = 0.80`.


In [ ]:
CORR_CLUSTER_THRESHOLD = 0.80
DIST_THRESHOLD = 1.0 - CORR_CLUSTER_THRESHOLD

corr_X = df_train_i[NUM_COLS_FULL].corr().abs().values.copy()
np.fill_diagonal(corr_X, 1.0)
dist = 1.0 - corr_X
np.fill_diagonal(dist, 0.0)
Z = linkage(squareform(dist, checks=False), method='average')
clusters = fcluster(Z, t=DIST_THRESHOLD, criterion='distance')
cluster_map = pd.Series(clusters, index=NUM_COLS_FULL, name='cluster')

fig, ax = plt.subplots(figsize=(13, 6))
dendrogram(Z, labels=NUM_COLS_FULL, leaf_rotation=90, color_threshold=DIST_THRESHOLD, ax=ax)
ax.axhline(DIST_THRESHOLD, color='red', ls='--', lw=1.5,
           label=f'порог |corr|={CORR_CLUSTER_THRESHOLD:.2f}')
ax.set_ylabel('1 − |corr|')
ax.set_title('Иерархическая кластеризация признаков (average linkage)')
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

print(f'Получено кластеров: {cluster_map.nunique()}')


## 2. Champion-логика внутри каждого кластера

Для champion-выбора нужны два критерия:
- `corr_y` = |Pearson(x, ln_price)| — связь с целью
- `communality_10PC` = Σ loadings² по первым 10 PC — насколько признак вписан в основную структуру X


In [ ]:
# corr_y
corr_y = df_train_i[NUM_COLS_FULL].corrwith(df_train_i['ln_price']).abs()

# communality по первым 10 PC
X_std = StandardScaler().fit_transform(df_train_i[NUM_COLS_FULL].values)
full_pca = PCA().fit(X_std)
L = full_pca.components_.T * np.sqrt(full_pca.explained_variance_)
L_df = pd.DataFrame(L, index=NUM_COLS_FULL,
                    columns=[f'PC{i+1}' for i in range(L.shape[1])])
n_90 = int(np.argmax(np.cumsum(full_pca.explained_variance_ratio_) >= 0.90) + 1)
comm10 = (L_df.iloc[:, :n_90]**2).sum(axis=1)

summary = pd.DataFrame({
    'corr_y':            corr_y.round(3),
    'communality_10PC':  comm10.round(3),
    'cluster':           cluster_map,
})

# Champion в каждом кластере = max corr_y
summary['is_champion'] = False
for c, grp in summary.groupby('cluster'):
    summary.loc[grp['corr_y'].idxmax(), 'is_champion'] = True

# Решение
def decide(row):
    if not row['is_champion']:
        return 'DROP (двойник)'
    if row['corr_y'] < 0.15 and row['communality_10PC'] < 0.90:
        return 'DROP (низкая полезность)'
    return 'KEEP'
summary['decision'] = summary.apply(decide, axis=1)

# Кого заменяет каждый DROP
champion_in_cluster = summary[summary['is_champion']].groupby('cluster').apply(lambda g: g.index[0])
summary['заменяется на'] = summary.apply(
    lambda r: champion_in_cluster[r['cluster']] if not r['is_champion'] else '—', axis=1)

view = pd.concat([
    summary[summary['decision']=='KEEP'].sort_values('corr_y', ascending=False),
    summary[summary['decision']!='KEEP'].sort_values('corr_y', ascending=False),
])
view


## 3. Решения по кластерам


In [ ]:
for c, grp in summary.groupby('cluster'):
    grp_sorted = grp.sort_values('corr_y', ascending=False)
    if len(grp_sorted) == 1:
        tag = 'одиночка'
    else:
        tag = f'{len(grp_sorted)} признаков, |corr| ≥ {CORR_CLUSTER_THRESHOLD:.2f}'
    print(f'[Кластер #{c}] {tag}:')
    for feat, row in grp_sorted.iterrows():
        flag = '★ CHAMPION' if row['is_champion'] else '         '
        print(f'  {flag}  {feat:<22} corr_y={row["corr_y"]:.3f}  '
              f'comm10={row["communality_10PC"]:.3f}  → {row["decision"]}')
    print()


## 4. Финальный список KEEP


In [ ]:
n_keep = (summary['decision']=='KEEP').sum()
n_drop = (summary['decision']!='KEEP').sum()
print(f'KEEP: {n_keep}   DROP: {n_drop}   (из {len(summary)})\n')

keep_list = summary[summary['decision']=='KEEP'].sort_values('corr_y', ascending=False).index.tolist()
drop_list = summary[summary['decision']!='KEEP'].index.tolist()

print('KEEP-список (сортировано по corr_y):')
for f in keep_list:
    print(f'  • {f}')
print('\nDROP-список:')
for f in drop_list:
    repl = summary.loc[f, 'заменяется на']
    print(f'  • {f}   (заменён на: {repl})')

# сохраним для следующих ноутбуков (через pickle/json)
import json as _json
with open('selected_features.json', 'w', encoding='utf-8') as fh:
    _json.dump({'keep': keep_list, 'drop': drop_list}, fh, ensure_ascii=False, indent=2)
print('\nСохранено в selected_features.json')


## Выводы блока 2

1. **5 признаков идут на выкид как двойники** в своих кластерах:
   - `front_track_width` ← `rear_track_width`
   - `width` ← `rear_track_width`
   - `length` ← `wheelbase`
   - `engine_power_hp` ← `max_torque_nm`
   - `cylinders_count` ← `engine_volume`
2. **Никто не выкинут по «низкой полезности»** — `trunk_volume` и `fuel_consumption_mixed` имеют `corr_y < 0.25`, но communality > 0.99 → они формируют независимые оси.
3. **Итог: 18 KEEP из 23.** Этот список сохранён в `selected_features.json` и будет использован в блоке 3.

→ В блоке 3 строим три модели на разных наборах признаков и сравниваем.
